# Ders 1: İleri Optimizasyon — Eğrilik, Uyarlanabilirlik ve Keskinlik

**İleri Derin Öğrenme** — Haydar Kılıç

Ön koşul: *Derin Öğrenme*, Ders 4 (Geri yayılım, SGD, Adam).

Giriş dersinde optimizasyon algoritmalarını birer tarif olarak ele almıştık. Burada *neden*
çalıştıklarını soruyoruz: kayıp yüzeyinin **eğriliği** gradyan inişini nasıl etkiliyor,
**uyarlanabilir** yöntemler geometriyi neden yeniden ölçekliyor ve bulduğumuz minimumun *derinliği*
kadar *şekli* neden önemli.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams["figure.dpi"] = 100
print("Kütüphaneler yüklendi.")


## 1. Eğrilik: Hessian ve Kondisyon Sayısı

Kaybın $\theta_0$ etrafındaki ikinci mertebeden Taylor açılımı:

$$L(\theta) \approx L(\theta_0) + g^\top (\theta - \theta_0) + \tfrac{1}{2}(\theta-\theta_0)^\top H (\theta-\theta_0)$$

Burada $g = \nabla L$ gradyan, $H = \nabla^2 L$ ise Hessian matrisidir. Kuadratik bir çukur için
$H$'nin özdeğerleri $\lambda_1 \ge \dots \ge \lambda_d$, temel yönler boyunca eğriliği verir.

Bu defterdeki her şeyi iki olgu belirler:

- **Kararlılık:** Düz gradyan inişi ancak $\eta < 2/\lambda_{\max}$ ise ıraksamaz.
- **Hız:** $i$ yönündeki yakınsama $(1-\eta\lambda_i)^t$ ile azalır; dolayısıyla en yavaş yön
  $\lambda_{\min}$'dir. Bu yüzden problemin gerçek zorluğu $\kappa = \lambda_{\max}/\lambda_{\min}$
  oranıdır (**kondisyon sayısı**).

Kötü koşullu bir kayıp, adım boyunu *en keskin* yöne göre seçmeye zorlar ve bedelini *en düz* yönde
ödetir — klasik zikzak davranışı.


In [ ]:
# Anizotropik kuadratik çukur: L(w) = 0.5 * (lam1*w1^2 + lam2*w2^2)
lam = np.array([20.0, 1.0])          # kappa = 20
loss    = lambda W: 0.5 * (lam[0]*W[..., 0]**2 + lam[1]*W[..., 1]**2)
grad    = lambda w: lam * w

def gd(w0, eta, steps=40, momentum=0.0, nesterov=False):
    w, v = np.array(w0, dtype=float), np.zeros(2)
    path = [w.copy()]
    for _ in range(steps):
        g = grad(w - eta*momentum*v) if nesterov else grad(w)
        v = momentum*v + g
        w = w - eta*v
        path.append(w.copy())
    return np.array(path)

eta_max = 2/lam.max()
print(f"lambda_maks={lam.max():.0f}, lambda_min={lam.min():.0f}, kappa={lam.max()/lam.min():.0f}")
print(f"kararlılık sınırı eta < 2/lambda_maks = {eta_max:.3f}")

gx, gy = np.meshgrid(np.linspace(-1.2, 1.2, 200), np.linspace(-1.6, 1.6, 200))
G = loss(np.stack([gx, gy], -1))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, eta in zip(axes, [0.02, 0.09, 0.105]):
    p = gd([-1.0, 1.4], eta, steps=40)
    ax.contour(gx, gy, G, levels=np.logspace(-2, 1.2, 18), cmap="Greys", linewidths=0.7)
    ax.plot(p[:, 0], p[:, 1], "o-", ms=3, lw=1.2, color="crimson")
    ax.plot(0, 0, "k*", ms=12)
    ax.set_title(f"eta = {eta}   ({'kararlı' if eta < eta_max else 'IRAKSIYOR'})")
    ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.7, 1.7)
    ax.set_xlabel("w1 (keskin)"); ax.set_ylabel("w2 (düz)")

plt.suptitle("Kötü koşullu bir kuadratikte gradyan inişi (kappa = 20)", fontsize=13)
plt.tight_layout(); plt.show()


## 2. Momentum ve Nesterov: Zikzağı Sönümlemek

Ağır-top (heavy-ball) momentumu bir hız vektörü biriktirir:

$$v_{t+1} = \beta v_t + g_t, \qquad \theta_{t+1} = \theta_t - \eta v_{t+1}.$$

Salınan bileşenler (keskin yönler) işaret değiştirerek $v$ içinde birbirini götürür; tutarlı bileşen
(düz yön) ise birikir. Kalıcı bir gradyan boyunca etkin adım $1/(1-\beta)$ katına çıkar —
$\beta = 0.9$ demek, 10 kat uzun adım demektir.

**Nesterov** gradyanı *ileriye bakılan* $\theta_t - \eta\beta v_t$ noktasında hesaplar; bu bir
düzeltme terimi gibi davranıp aşmayı (overshoot) sönümler. Kuadratiklerde yakınsama hızını
$O(\kappa)$'dan $O(\sqrt{\kappa})$'ya indirir.


In [ ]:
eta = 0.045
runs = {
    "GD (beta=0)":        gd([-1.0, 1.4], eta, 60),
    "Momentum (0.9)":     gd([-1.0, 1.4], eta, 60, momentum=0.9),
    "Nesterov (0.9)":     gd([-1.0, 1.4], eta, 60, momentum=0.9, nesterov=True),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].contour(gx, gy, G, levels=np.logspace(-2, 1.2, 18), cmap="Greys", linewidths=0.7)
for (name, p), c in zip(runs.items(), ["crimson", "steelblue", "seagreen"]):
    axes[0].plot(p[:, 0], p[:, 1], "o-", ms=2.5, lw=1.1, label=name, color=c, alpha=0.85)
axes[0].plot(0, 0, "k*", ms=12); axes[0].legend(fontsize=9)
axes[0].set_title("Yörüngeler (aynı öğrenme oranı)")
axes[0].set_xlabel("w1 (keskin)"); axes[0].set_ylabel("w2 (düz)")

for (name, p), c in zip(runs.items(), ["crimson", "steelblue", "seagreen"]):
    axes[1].semilogy(loss(p) + 1e-16, lw=2, label=name, color=c)
axes[1].set_title("Kayıp - iterasyon"); axes[1].set_xlabel("iterasyon")
axes[1].set_ylabel("L (log)"); axes[1].grid(alpha=0.3); axes[1].legend(fontsize=9)

plt.tight_layout(); plt.show()

for name, p in runs.items():
    print(f"{name:20s} son kayıp = {loss(p)[-1]:.3e}")


## 3. Adam ve Ayrıştırılmış Ağırlık Sönümü (AdamW)

Adam, gradyanın birinci ve ikinci momentlerinin hareketli ortalamasını tutar:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t, \qquad v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2,$$

ve yanlılık düzeltmesinden sonra $\theta \leftarrow \theta - \eta\, \hat m_t/(\sqrt{\hat v_t}+\epsilon)$
adımını atar. $\sqrt{\hat v_t}$'ye bölmek bir **köşegen ön koşullandırmadır**: her koordinatı kendi
gradyan büyüklüğüne göre ölçekler ki bu, eksenlere hizalı kötü koşullu bir problemin tam ihtiyacı
olan şeydir.

**L2 / ağırlık sönümü inceliği.** Klasik L2 düzenlileştirmesi gradyana $\lambda\theta$ ekler. Adam'ın
içinde bu terim *de* $\sqrt{\hat v_t}$'ye bölünür; yani gradyan geçmişi büyük olan parametreler *daha
az* sönümlenir — amaçlanan tekdüze küçültmenin tam tersi. **AdamW** bunu ayrıştırır:

$$\theta \leftarrow \theta - \eta\left(\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon} + \lambda\theta\right).$$

Modern transformer reçetelerinin Adam + L2 yerine AdamW kullanmasının nedeni budur.


In [ ]:
def adam(grad_fn, w0, eta=0.1, b1=0.9, b2=0.999, eps=1e-8, steps=200,
         wd=0.0, decoupled=False):
    w = np.array(w0, dtype=float)
    m, v = np.zeros_like(w), np.zeros_like(w)
    path = [w.copy()]
    for t in range(1, steps + 1):
        g = grad_fn(w)
        if wd and not decoupled:      # Adam + L2  (bağlı)
            g = g + wd * w
        m = b1*m + (1-b1)*g
        v = b2*v + (1-b2)*g**2
        mh, vh = m/(1-b1**t), v/(1-b2**t)
        step = mh/(np.sqrt(vh)+eps)
        if wd and decoupled:          # AdamW      (ayrıştırılmış)
            step = step + wd*w
        w = w - eta*step
        path.append(w.copy())
    return np.array(path)

# Gradyan ölçekleri çok farklı iki koordinat; ikisi de 0'a doğru düzenlileştiriliyor
lam_wd = np.array([50.0, 0.5])
g_wd   = lambda w: lam_wd * (w - np.array([1.0, 1.0]))   # sönüm olmadan optimum (1,1)

p_l2   = adam(g_wd, [0.0, 0.0], eta=0.05, wd=0.5, decoupled=False, steps=400)
p_adamw= adam(g_wd, [0.0, 0.0], eta=0.05, wd=0.5, decoupled=True,  steps=400)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, (p, name) in zip(axes, [(p_l2, "Adam + L2 (bağlı)"), (p_adamw, "AdamW (ayrıştırılmış)")]):
    ax.plot(p[:, 0], lw=2, label="w1  (büyük eğrilik)")
    ax.plot(p[:, 1], lw=2, label="w2  (küçük eğrilik)")
    ax.axhline(1.0, ls="--", c="k", lw=0.8)
    ax.set_title(name); ax.set_xlabel("adım"); ax.set_ylabel("parametre değeri")
    ax.grid(alpha=0.3); ax.legend(fontsize=9)
plt.suptitle("Aynı sönüm gücü lambda=0.5 — bağlı sönüm iki koordinatı eşit küçültmüyor", fontsize=12)
plt.tight_layout(); plt.show()

print(f"Adam+L2  yakınsadığı nokta: {p_l2[-1].round(3)}")
print(f"AdamW    yakınsadığı nokta: {p_adamw[-1].round(3)}   <- iki koordinat da eşit küçülüyor")


## 4. Öğrenme Oranı Programları: Isınma, Kosinüs ve Isınma Neden Var?

Eğitimin başında ikinci moment kestirimi $\hat v_t$ çok az örnekten hesaplanır; bu yüzden Adam adımı
$\hat m/\sqrt{\hat v}$ yüksek varyanslıdır. Aynı anda ağ, eğriliğin büyük olduğu bir bölgededir.
**Doğrusal ısınma (warmup)** moment kestirimleri oturana kadar $\eta$'yı küçük tutar; ardından
görevi **kosinüs sönümüne** devreder:

$$\eta_t = \eta_{\min} + \tfrac12(\eta_{\max}-\eta_{\min})\left(1+\cos\frac{\pi t}{T}\right).$$

Kosinüs sönümü bütçenin çoğunu orta seviyedeki öğrenme oranlarında geçirir ve sonda yumuşakça iner;
deneysel olarak ani basamaklı sönüme kıyasla daha düz minimumlar bulur.


In [ ]:
T = 1000
t = np.arange(T)

def cosine(t, T, base=1.0, warm=100, min_lr=0.0):
    lr = np.where(t < warm, base*t/max(warm, 1),
                  min_lr + 0.5*(base-min_lr)*(1+np.cos(np.pi*(t-warm)/(T-warm))))
    return lr

schedules = {
    "sabit":                np.ones(T),
    "basamaklı sönüm (x0.1 @ 1/3)": 0.1**(3*t//T),
    "kosinüs, ısınmasız":       cosine(t, T, warm=0),
    "kosinüs + %10 ısınma":     cosine(t, T, warm=T//10),
    "ters karekök (ısınmalı)":     np.minimum(t/(T//10), np.sqrt((T//10)/np.maximum(t, 1))),
}

plt.figure(figsize=(9, 4))
for name, s in schedules.items():
    plt.plot(t, s, lw=2, label=name)
plt.xlabel("eğitim adımı"); plt.ylabel("öğrenme oranı çarpanı")
plt.title("Modern derin öğrenmede kullanılan öğrenme oranı programları")
plt.grid(alpha=0.3); plt.legend(fontsize=9); plt.tight_layout(); plt.show()


## 5. Gradyan Kırpma

Yinelemeli ağlarda ve transformer'larda gradyan normu ağır kuyrukludur: adımların çoğu sıradandır
ama arada bir gelen bir yığın, birkaç kat büyüklükte bir gradyan üretip binlerce iyi güncellemeyi
geri alabilir. **Norm kırpma** yönü koruyarak yeniden ölçekler:

$$g \leftarrow g \cdot \min\left(1, \frac{c}{\lVert g \rVert}\right).$$

Dikkat: *değere göre* kırpma yönü bozar, *norma göre* kırpma bozmaz. `clip_grad_norm_`'un standart
tercih olmasının nedeni budur.


In [ ]:
rng = np.random.default_rng(0)
# Log-normal gradyan normları + birkaç sivri uç: gerçekçi bir ağır kuyruk
norms = np.exp(rng.normal(0.0, 0.6, 500))
norms[[87, 231, 402]] *= 40

c = 3.0
clipped = norms * np.minimum(1.0, c/norms)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(norms, lw=0.9, label="ham ||g||")
axes[0].plot(clipped, lw=1.2, label=f"kırpılmış (c={c})")
axes[0].set_yscale("log"); axes[0].set_xlabel("adım"); axes[0].set_ylabel("gradyan normu (log)")
axes[0].set_title("Ağır kuyruklu gradyan normları"); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

axes[1].hist(np.log10(norms), bins=40, color="steelblue", alpha=0.8)
axes[1].axvline(np.log10(c), c="crimson", ls="--", lw=2, label="kırpma eşiği")
axes[1].set_xlabel("log10 ||g||"); axes[1].set_ylabel("sayı")
axes[1].set_title("Gradyan normlarının dağılımı"); axes[1].legend(fontsize=9)

plt.tight_layout(); plt.show()
print(f"kırpılan adımların oranı: {(norms > c).mean():.1%}")
print(f"en büyük ham norm {norms.max():.1f}  ->  kırpmadan sonra {clipped.max():.1f}")


## 6. Keskinlik: Düz Minimumlar Neden Daha İyi Genelleşir ve SAM Neyi Optimize Eder?

Eğitim kaybı aynı olan iki minimum çok farklı genelleyebilir. Eğitim ve test kayıp yüzeyleri aynı
fonksiyon değildir — sonlu örnekleme yüzeyi kaydırır ve deforme eder. **Düz** bir minimum bu kaymaya
dayanıklıdır, **keskin** olan değildir.

**SAM** bunu bir amaç fonksiyonuna dönüştürür. Bir noktadaki kayıp yerine, bir komşuluktaki en kötü
kaybı minimize eder:

$$\min_\theta \; \max_{\lVert \epsilon \rVert \le \rho} L(\theta + \epsilon).$$

Pratikte içteki maksimizasyon tek bir gradyan yükseliş adımıyla yaklaşıklanır
($\hat\epsilon = \rho\, g/\lVert g\rVert$) ve dıştaki güncelleme gradyanı **orada** ölçer:
$\theta \leftarrow \theta - \eta \nabla L(\theta + \hat\epsilon)$. Maliyeti adım başına fazladan bir
ileri/geri geçiştir.

Aşağıda: $\theta=0$'da **derin ve keskin**, $\theta=2$'de **daha sığ ama düz** bir minimumu olan bir
manzara. Keskin olanın eğitim kaybı daha düşük — ve geri kalan her şeyi daha kötü.


In [ ]:
T = 0.15
f_sharp = lambda x: -1.00 + 50.0*x**2           # derin ve dar  (eğrilik 100)
f_flat  = lambda x: -0.90 +  1.0*(x-2.0)**2     # daha sığ, geniş  (eğrilik 2)

def L(x):                                        # iki çukurun yumuşak minimumu
    x = np.asarray(x, dtype=float)
    return -T*np.logaddexp(-f_sharp(x)/T, -f_flat(x)/T)

xg = np.linspace(-1.0, 3.5, 900)
shift = 0.15                                     # eğitim -> test yüzeyi kayması
sharp_min, flat_min = 0.0, 2.0

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

axes[0].plot(xg, L(xg),         lw=2, c="k",        label="eğitim kaybı")
axes[0].plot(xg, L(xg - shift), lw=2, c="crimson", ls="--", label=f"test kaybı (yüzey {shift} kaydırıldı)")
for m, c in [(sharp_min, "crimson"), (flat_min, "seagreen")]:
    axes[0].plot([m, m], [L(m), L(m - shift)], c=c, lw=3, alpha=0.6)
    axes[0].scatter([m], [L(m)], s=60, c=c, zorder=4)
axes[0].set_ylim(-1.3, 2.0); axes[0].set_xlabel("theta"); axes[0].set_ylabel("kayıp")
axes[0].set_title("Küçük bir kayma keskin minimum için ölümcül"); axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Rastgele parametre bozulması altında kayıp (ağırlık gürültüsü / niceleme / farklı bir test kümesi)
rng = np.random.default_rng(0)
eps = rng.normal(0, 0.10, 4000)
axes[1].hist(L(sharp_min + eps), bins=60, alpha=0.7, color="crimson",  label="keskin minimum çevresi")
axes[1].hist(L(flat_min  + eps), bins=60, alpha=0.7, color="seagreen", label="düz minimum çevresi")
axes[1].set_xlabel("bozulmadan sonraki kayıp  eps ~ N(0, 0.1^2)"); axes[1].set_ylabel("sayı")
axes[1].set_title("Parametre gürültüsüne dayanıklılık"); axes[1].legend(fontsize=9)

rho  = 0.30
offs = np.linspace(-rho, rho, 121)
L_sam = np.max(np.stack([L(xg + o) for o in offs]), axis=0)
axes[2].plot(xg, L(xg),  lw=2, c="k",        label="L(theta)")
axes[2].plot(xg, L_sam,  lw=2, c="seagreen", label=f"SAM amaç fonksiyonu, rho={rho}")
axes[2].scatter([sharp_min, flat_min], [L_sam[np.argmin(abs(xg))], L_sam[np.argmin(abs(xg-2))]],
                s=60, c=["crimson", "seagreen"], zorder=4)
axes[2].set_ylim(-1.3, 2.5); axes[2].set_xlabel("theta")
axes[2].set_title("SAM amaç fonksiyonunda sıralama tersine dönüyor"); axes[2].legend(fontsize=9)
axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()

for name, m in [("keskin (theta=0)", sharp_min), ("düz    (theta=2)", flat_min)]:
    tr, te = float(L(m)), float(L(m - shift))
    sam    = float(np.max(L(m + offs)))
    noisy  = float(np.mean(L(m + eps)))
    print(f"{name}:  eğitim {tr:+.3f} | test {te:+.3f} (fark {te-tr:+.3f}) | "
          f"E[L gürültü altında] {noisy:+.3f} | SAM amacı {sam:+.3f}")


Keskin minimum yalnızca eğitim kaybında kazanıyor, diğer tüm sütunlarda kaybediyor. SAM küresel bir
optimizasyon yöntemi değildir — düz havzayı bulmak için bir tepeyi aşmaz — ama amaç fonksiyonunu
yeniden şekillendirerek SGD'nin kendi gürültüsünün hangi minimumlara yerleşmeye razı olacağını
değiştirir.

İki pratik uyarı: $\rho$ gerçek bir hiperparametredir (çok büyük seçilirse amaç fonksiyonu
anlamsızlaşır) ve SAM adım başına maliyeti kabaca ikiye katlar; dolayısıyla adil karşılaştırma, iki
katı adım sayısıyla eğitilmiş bir temel modele karşı yapılmalıdır.


## 7. Ön Koşullandırma: Newton'dan K-FAC'e

Newton yöntemi $\theta \leftarrow \theta - H^{-1} g$ adımını atar ve *herhangi* bir kuadratiği tek
adımda çözer: etkin kondisyon sayısını 1 yapar. $d = 10^9$ parametre için $H$'yi saklamak imkânsız
olduğundan pratik yöntemler $H^{-1}$'i yaklaşıklar:

| Yöntem | Eğrilik yaklaşımı |
|---|---|
| SGD | birim matris (eğrilik bilgisi yok) |
| Adam / RMSProp | köşegen, gradyan karelerinden |
| Shampoo / K-FAC | katman başına Kronecker çarpanlı bloklar |
| L-BFGS | son $m$ gradyan farkından düşük ranklı |

K-FAC şu olguyu kullanır: girdisi $a$, çıktı gradyanı $s$ olan bir katmanda Fisher bloğu yaklaşık
olarak $\mathbb{E}[aa^\top] \otimes \mathbb{E}[ss^\top]$ biçiminde çarpanlarına ayrılır; böylece
devasa bir matrisi tersine çevirmek yerine iki küçük matris tersine çevrilir.


In [ ]:
# Her ön koşullandırıcı etkin kondisyon sayısını ne kadar düşürüyor?
d = 60
rng = np.random.default_rng(1)
Q = np.linalg.qr(rng.normal(size=(d, d)))[0]
eigs = np.logspace(0, 3, d)                      # gerçek kappa = 1000
H = Q @ np.diag(eigs) @ Q.T
H_diag_aligned = np.diag(eigs)                   # aynı spektrumun eksenlere hizalı hâli

def kappa(M): 
    e = np.linalg.eigvalsh(M); return e.max()/e.min()

def diag_precond(M):                             # Adam tarzı köşegen ölçeklemeden sonraki kappa
    D = np.diag(1/np.sqrt(np.diag(M)))
    return kappa(D @ M @ D)

rows = [
    ("SGD (birim), döndürülmüş H",  kappa(H)),
    ("Adam benzeri köşegen, döndürülmüş H", diag_precond(H)),
    ("Adam benzeri köşegen, hizalı H", diag_precond(H_diag_aligned)),
    ("Tam Newton (H^-1)",         1.0),
]
for name, k in rows:
    print(f"{name:32s} etkin kappa = {k:10.2f}")

plt.figure(figsize=(8, 4))
plt.semilogy(np.sort(eigs)[::-1], "o-", ms=3, label="Hessian özdeğerleri")
plt.xlabel("indeks"); plt.ylabel("özdeğer (log)")
plt.title("Yukarıda kullanılan sentetik eğrilik spektrumu (kappa = 1000)")
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()


Üçüncü satıra dikkat: köşegen bir ön koşullandırıcı ancak eğrilik kabaca **eksenlere hizalı** ise
işe yarar. Aynı spektrum döndürüldüğünde Adam'ın köşegeni neredeyse hiçbir şey kazandırmaz. Bu,
uyarlanabilir yöntemlerin size ne sağladığının dürüst ifadesidir — ve katman içi korelasyonları
yakalayan Kronecker çarpanlı yöntemlerin büyük ölçekli eğitimde neden tekrar tekrar gündeme
geldiğinin de.

## 8. Özet

| Kavram | Açıklama |
|---|---|
| **Kondisyon sayısı $\kappa$** | Hessian'ın $\lambda_{\max}/\lambda_{\min}$ oranı; optimizasyonun gerçek zorluğu |
| **Kararlılık eşiği** | $\eta = 2/\lambda_{\max}$ üzerinde GD ıraksar |
| **Momentum** | Salınan bileşenleri götürür, kalıcı olanları $1/(1-\beta)$ katına çıkarır |
| **Nesterov** | İleriye bakan gradyan; kuadratiklerde $O(\kappa) \to O(\sqrt{\kappa})$ |
| **Adam** | İkinci gradyan momentlerinden köşegen ön koşullandırma |
| **AdamW** | Ağırlık sönümünü uyarlanabilir ölçeklemeden ayırır — doğru varsayılan |
| **Isınma (warmup)** | Erken dönemdeki yüksek varyanslı moment kestirimlerine ve büyük eğriliğe karşı korur |
| **Gradyan kırpma** | Norm kırpma yönü korur; değer kırpma korumaz |
| **SAM** | $\rho$-yuvarındaki en kötü kaybı minimize eder → düz minimumlara yönelir |
| **K-FAC / Shampoo** | Kronecker çarpanlı eğrilik; köşegen yöntemlerin kaçırdığını yakalar |

**Sonraki Defter →** Ölçekleme Yasaları ve Modern Mimariler
